## Setup

In [1]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

26/05/08 11:32:32 WARN Dispatcher: Message RequestMessage(172.18.0.7:43720, NettyRpcEndpointRef(spark://CoarseGrainedScheduler@3bb24d973e4f:39119), RemoveExecutor(1,Unable to create executor due to Exception thrown in awaitResult: )) dropped due to sparkEnv is stopped. Could not find CoarseGrainedScheduler.


In [2]:
workshop_name = "file-size-deep-dive"

from pyspark.sql import SparkSession

executor_memory = "8g"
executor_cores = 2
num_executors = 2


spark = SparkSession.builder \
        .appName(workshop_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
    .getOrCreate()

Setting Spark log level to "WARN".


* Open [http://localhost:4040](http://localhost:4040) to see your Spark UI

In [ ]:
# Create Fake Data 
import os 
import glob
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

################################################################################
## CHANGE DATA SIZE AS NEEDED
################################################################################

data_size_gb = 30
overwrite_existing_data = False # CHANGE TO TRUE TO CREATE DATA
data_folder = './data'

# Only when overwrite = False and there are atleast 1 csv files we need to skip data gen (everyother time we need to generate data) 
if not (not overwrite_existing_data and len(glob.glob(os.path.join(data_folder, '*.csv'))) > 1):
    print(f'Creating raw TPCH data in {data_folder}')
    run(scaling_factor=data_size_gb)

print(f'Loading data in raw TPCH folder {data_folder} to Iceberg Tables')
run_ddl(data_path=Path(data_folder), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

In [3]:
%%sql
use prod.db

26/05/08 11:33:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


++
||
++
++

### Configuration

| Parameter | Value |
|-----------|-------|
| Apache Spark version | 4.0.2 |
| Apache Iceberg version | 1.10.0 |
| Number of Executors | 2 |
| Executor memory | 8GB |
| Cores per executor | 2 |
| Data size | 30GB |

## Spark should focus on data processing, not opening multiple small files

* Let's look at an example

In [4]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM
  lineitem
GROUP BY
  1
ORDER BY
  2 desc
LIMIT
  10

receipt_month,num_line_items
7,16234226
8,16161841
5,16140024
6,15710469
9,15195007
4,15141057
10,15116206
3,15056278
11,14054000
12,14015957


![Small File IO](images/small_file_io.png)

### Short green chunks in Stages tab, you've got too many small files

* In the read stage of the above group by query you will see multiple small green chunks.
* The green chunks indicate data processing time and should be longer.
![Small Files Task View](images/small_files_task.png)

## Use helper function to resize files

* Table formats have helper functions that you can use to resize your files to be of optimal sizes.
* An optimal size is anywhere from 512MB to 1GB, depending on your executor memory.
* **Note** that data in disk will expand in memory, as data is stored as parquet on disk.
* Let's see how we can resize our data with `rewrite_data_files` Iceberg function. 

In [ ]:
%%sql
CREATE TABLE
  prod.db.lineitem_resized AS
SELECT
  *
FROM
  prod.db.lineitem

In [ ]:
spark.sql("CALL demo.system.rewrite_data_files('prod.db.lineitem_resized')")
# resizes files in the table to 512 MB
# has options to resize per partition, when running this as part of your pipeline
# Delta has OPTIMIZE, which is similar to this Iceberg function

* Here is what happens

![Rewrite Files](images/rewrite_files.png)

* Now let's re-run the `group by` from before to see the performance?

In [5]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM
  lineitem_resized
GROUP BY
  1
ORDER BY
  2 desc
LIMIT
  10

receipt_month,num_line_items
7,16234226
8,16161841
5,16140024
6,15710469
9,15195007
4,15141057
10,15116206
3,15056278
11,14054000
12,14015957


### Optimal file sizes let's spark focus on data processing

![Large File IO](images/large_file_io.png)

* In stage tab you can see how much better this process is.
![Large File task](images/large_files_task.png)

## Use table properties to size files optimally

* Data is written to an Iceberg table with an Iceberg writer task.
* We can use settings to define how Iceberg writer tasks writes to disk.
* Let's look at an example

In [6]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_512mb_part_size (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg TBLPROPERTIES (
    'format-version' = '2',
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

++
||
++
++

In [9]:
%%sql
INSERT INTO
  prod.db.lineitem_512mb_part_size
SELECT
  *
FROM
  prod.db.lineitem

++
||
++
++

* The propery `write.spark.advisory-partition-size-bytes` indicates the size of data in memory before Spark-Iceberg writer inserts data into this table.
* Let's take a look at our data files and see if our files are about 500MB.
* We can use the query below to see the file sizes or see storage at [http://localhost:9001](http://localhost:9001), with username and password as `admin` and `password` respectively.

In [10]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM
  prod.db.lineitem_512mb_part_size.files
ORDER BY
  2, 3

26/05/08 12:06:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 12:06:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 12:06:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 12:06:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 12:06:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_mb,writer_task
1,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00000-413-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.9,413
2,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00001-414-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,414
3,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00002-415-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,415
4,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00003-416-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,416
5,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00004-417-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,417
6,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00005-418-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,418
7,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00006-419-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,419
8,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00007-420-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.68,420
9,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00008-421-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.66,421
10,s3://warehouse/prod/db/lineitem_512mb_part_size/data/00009-422-0ba91d58-64cf-4652-ae96-83e99761fb00-0-00001.parquet,26.63,422


* The data sizes for the above file are smaller than what we expected.
* Iceberg uses a `spark.sql.iceberg.advisory-partition-size` and sets it to 384MB by default. ([ref](https://github.com/apache/iceberg/issues/10051)).
* The 384MB represents size in-memory, when we write out to parquet we can expect 3-10x compression (depending on your data).
![Compression](images/compression.png)                                                  * With this in mind we can set `write.spark.advisory-partition-size-bytes` to a significantly large number. 
* Note that we need a large enough executor to enable us to do this.


### Ensure your executor has the memory for this

In [11]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_512mb_part_size_2gb_partsize (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648', -- 2 GB
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

++
||
++
++

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_512mb_part_size_2gb_partsize
SELECT
  *
FROM
  prod.db.lineitem

[Stage 16:>                                                       (0 + 4) / 180]

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM
  prod.db.lineitem_512mb_part_size_2gb_partsize.files
ORDER BY
  2, 3

* We can see now that the files are of optimal size

## Partitioning can help, but only if you repartition first

* In case of partitioned tables, one might assume that Spark will appropriately size the data per partition, but you'd be wrong.
* Let's look at an example.

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648', -- 2 GB
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year_2gb_intask_mem
SELECT
  *
FROM
  prod.db.lineitem

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM
  prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY
  2, 3

### Iceberg writer writes incoming data to its partition

* What's happening here is that the data from `lineitem` table is not partitioned when it arrives in the iceberg write task.
* An iceberg write task got input for year 1997 & 1996 so it only had enough data for ~200MB and ~180MB
![Iceberg Task Write](images/iceberg_write.png)
* In the task ui we can see how muiltiple tasks are writing data to multiple partitions.
* **Note** In the above query we use `split(file_path, '-')[1] AS writer_task` as the second part of the file name indicates the task id that wrote that partition.
![Iceberg Task Write w Spark Partition](images/task_file_map_no_part.png)
* If we partition the data in Spark before writing to disk, we can avoid this issue. Let's see how.

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem_code_part (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648', -- 2 GB
    'write.target-file-size-bytes' = '536870912' -- 512 MB
  );

In [ ]:
# This forces same-year rows to same task
import pyspark.sql.functions as F

spark.table("prod.db.lineitem")\
  .repartition(F.year(F.col("l_shipdate"))) \
  .writeTo("prod.db.lineitem_part_year_2gb_intask_mem_code_part") \
  .append()

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM
  prod.db.lineitem_part_year_2gb_intask_mem_code_part.files
ORDER BY
  2, 3

### Partition data before writing for optimal file size

![Iceberg Write](images/spark_write_part.png)

* Iceberg writer overflow data into a second file, when the file gets bigger than 512MB `write.target-file-size-bytes`

![Iceberg Task Write w Spark Partition](images/task_file_map_part.png)

### Alternatives and future work

* We can also use `write.distribution.mode` as Range (which sorts data) or sort data in spark before writing to get to optimal file size.
* As of Iceberg version `1.10.0`: "Future work in Spark should allow Iceberg to automatically adjust this (spark.sql.adaptive.advisoryPartitionSizeInBytes) parameter at write time to match the write.target-file-size-bytes." [Iceberg docs](https://iceberg.apache.org/docs/1.10.0/spark-writes/#controlling-file-sizes)

## Vendors do this (& more) for free

* Data maintenance is a place where vendors do a great job.
* Both Snowflake and Databricks have automated file resizing (along with cleaning up catalogs, deleting old files, etc).
* If you don't want to do maintenance (or want much simpler management in general) vendors are a good choice.

| Concern | Databricks Unity Catalog | Snowflake |
|---|---|---|
| File compaction | ✅ Predictive Optimization | ✅ automatic |
| Old snapshot cleanup | ✅ automatic | ✅ automatic |
| Orphan file removal | ✅ automatic | ✅ automatic |
| Manifest compaction | N/A | ✅ automatic |

* Both dbx and Snowflake have sane defaults for data retention (which dictates file removal) which you can change as needed.
* I recently saw an issue talking about [this on Reddit](https://www.reddit.com/r/dataengineering/comments/1t66thf/how_are_you_guys_handling_iceberg_table/)